# 00. Gobierno, contexto y definición del problema + diseño y reproducibilidad

**Fases del guía metodológica cubiertas: 0 (Problema y contexto), 1 (Diseño y reproducibilidad), 2 (Gobierno de datos)**

> Regla central aplicada: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.
> El test nunca influye en preprocessing, selección de variables, hiperparámetros o elección de modelo.



## 0.1 Contexto

- **Problema**: predecir el riesgo de consumo alto de alcohol en fin de semana de
  estudiantes de secundaria portugueses, a partir de características sociodemográficas,
  familiares y académicas.
- **Usuario final**: orientadores escolares y equipos de salud escolar que priorizan
  programas de prevención.
- **Proceso actual sin automatización**: detección reactiva (tras incidentes) o encuestas manuales
  sin priorización objetiva.
- **Decisión que apoyará el modelo**: qué alumnos deberían recibir intervención preventiva
  (charlas, seguimiento) con un presupuesto limitado.
- **Valor esperado**: priorizar intervenciones en el ~35-40 % de alumnos con consumo alto.
- **Coste y restricciones**: coste de cada intervención (tiempo del orientador);
  restricciones de privacidad (menores de edad, datos familiares).
- **Alternativas no basadas en ML**: reglas simples (ej. "goout >= 4 y Dalc >= 3 -> alerta"),
  heurísticas o screening manual. El modelo debe superarlas en precisión de priorización.

## 0.2 Formulación técnica

- **Clasificación binaria supervisada** (tabla de datos tabular).
- Target: `Walc` (consumo de alcohol en fin de semana, escala 1-5) binarizado:
  **alto = Walc >= 3**.
- No aplican visión, NLP, series temporales, ranking, RL, etc. (dataset tabular pequeño:
  382 alumnos tras merge).

## 0.3 Definición operativa

- **Unidad de predicción**: alumno.
- **Entrada disponible en producción**: cuestionario sociodemográfico (sin calificaciones
  G1/G2/G3, que son post-evento).
- **Salida esperada**: probabilidad de consumo alto + clase binaria.
- **Horizonte**: predicción a inicio de curso (las features se conocen antes del resultado).
- **Frecuencia**: batch anual (cada cohorte nueva), no tiempo real.
- **Coste de errores**: FN (no intervenir a un alumno de riesgo) más grave que FP
  (intervención innecesaria) -> matriz de costes FP=1, FN=2.
- **Límites**: latencia no crítica (batch), interpretabilidad importante (orientadores),
  sin datos personales identificativos.

## 1.1 Estructura del proyecto

```
project/
├── README.md, .gitignore, requirements.txt, LICENSE
├── configs/            # config.yaml, experiments.yaml
├── data/raw|interim|processed
├── notebooks/          # 00..17 (una por fase o grupo de fases)
├── src/data|features|models|evaluation|api
├── models/             # artefactos .joblib
├── reports/figures/    # gráficos y JSON de métricas
├── tests/              # pytest
├── docs/               # model card, informe técnico, API
└── scripts/            # run_pipeline.py, build_notebooks_*.py
```

## 1.2 Reproducibilidad

- Python 3.11, semillas fijadas en `configs/config.yaml` (`random_state: 42`).
- `requirements.txt` con versiones mínimas.
- Datos inmutables en `data/raw/`; transformaciones reproducibles en `src/`.
- Registro de experimentos en `reports/experiments.csv`.



### 1.2.1 Lectura de la configuración del proyecto

Cargamos el fichero `configs/config.yaml`, que es la **fuente única de verdad** del
proyecto: contiene la semilla maestra, la definición del target, los tamaños de la
partición, el número de folds de validación cruzada y la métrica primaria. Imprimimos
esos valores para dejar constancia de cuáles son los parámetros que gobiernan todo el
experimento y que **no deben modificarse** una vez iniciado el flujo (regla de oro del
control experimental).


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
cfg = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text(encoding="utf-8"))
print("Semilla:", cfg["project"]["random_state"])
print("Target:", cfg["target"])
print("Split:", cfg["split"])
print("CV folds:", cfg["model"]["cv_folds"], "| métrica primaria:", cfg["model"]["scoring_primary"])


Semilla: 42
Target: {'name': 'Walc', 'positive_label': 1, 'threshold': 3, 'meaning': 'alta_frecuencia_consumo_fin_semana', 'post_event_columns': ['G1', 'G2', 'G3']}
Split: {'strategy': 'stratified', 'val_size': 0.2, 'test_size': 0.2}
CV folds: 5 | métrica primaria: roc_auc



### 1.2.2 Versiones del entorno de ejecución

Para garantizar la **reproducibilidad** del experimento necesitamos conocer las versiones
exactas de Python y de las librerías científicas con las que se ejecuta este notebook.
Si alguien intentara reproducir el proyecto en otro entorno, estas versiones le permitirían
detectar diferencias (por ejemplo, un cambio de comportamiento en `pandas` o `scikit-learn`)
que podrían explicar resultados distintos. Imprimimos las versiones de las librerías
principales: `numpy`, `pandas`, `scikit-learn`, y los tres potenciadores de gradiente
(`xgboost`, `lightgbm`, `catboost`) que se compararán en la fase 12.


In [2]:

import sys, platform, numpy, pandas, sklearn
print("Python   :", platform.python_version())
print("numpy    :", numpy.__version__)
print("pandas   :", pandas.__version__)
print("sklearn  :", sklearn.__version__)
import xgboost, lightgbm, catboost
print("xgboost  :", xgboost.__version__)
print("lightgbm :", lightgbm.__version__)
print("catboost :", catboost.__version__)


Python   : 3.11.9
numpy    : 2.4.6
pandas   : 3.0.5
sklearn  : 1.9.0


xgboost  : 3.2.0
lightgbm : 4.7.0
catboost : 1.2.10



## 2.1 Obtención

- **Fuente**: UCI Machine Learning Repository — dataset *Student Performance*
  (P. Cortez y A. Silva, 2008). Copia obtenida de Kaggle
  (`uciml/student-alcohol-consumption`).
- **Método**: descarga del ZIP `data/raw/archive.zip` -> CSV por coma (en la copia de
  Kaggle; en UCI el separador es `;` — detectado en auditoría).
- **Periodo**: curso 2005-2006, dos escuelas portuguesas (Gabriel Pereira GP y Mousinho
  da Silveira MS).
- **Integridad**: 395 filas (matemáticas) + 649 filas (portugués) de cuestionario,
  que corresponden a **662 alumnos únicos** (cada alumno aparece una vez por asignatura
  cursada). El merge interno del paper (`student-merge.R`) conserva solo los 382 que
  cursan ambas asignaturas; este proyecto usa el **dataset ampliado de 662 alumnos
  únicos** (1 fila por alumno, priorizando Matemáticas) para maximizar la muestra de
  forma coherente (ver `docs/dataset_estructura.md`).

## 2.2 Legalidad y uso

- Licencia de la copia de Kaggle: **CC BY 4.0** (atribución requerida, uso académico/comercial).
- Dataset original UCI: *Cortez, P. & Silva, A. (2008). Using Data Mining to Predict
  Secondary School Student Performance. Proceedings of 5th FUture BUsiness TEChnology
  Conference*. 
- **Atribución obligatoria** en README y documentación.
- Datos de menores de edad: el dataset es anónimo (sin nombres, IDs ni datos
  identificativos); se trata con fines académicos de predicción de riesgo, no de diagnóstico.

## 2.3 Etiquetas y calidad

- **Origen del target**: auto-reporte del alumno en cuestionario (consumo en día laborable
  `Dalc` y fin de semana `Walc`, escala 1-5).
- **Criterio**: definición operativa de "consumo alto" = `Walc >= 3` (frecuencia 3-5 días).
- **Ambigüedad / ruido**: auto-reporte -> posible sesgo de deseabilidad social; documentado.
- **Sin anotadores externos**: etiqueta auto-declarada, sin acuerdo entre anotadores.
- **Retraso de etiqueta**: la etiqueta corresponde al mismo curso; en producción se
  obtendría por encuesta al inicio del curso siguiente.
